# Capítulo 7 — Pós-recuperação (Retrieval Post-Processing)

**Livro:** começa na página **348** do PDF.

📗 **Gabarito** dos exercícios do caderno de prática. Tente sozinho antes de olhar.

## Preparação

In [ ]:
import sys
sys.path.insert(0, "../comum")  # ia.py (modelos via OpenRouter) e corpus.py (base de conhecimento de exemplo)
from ia import *
from corpus import DOCS
import re, json, math, random
import numpy as np
print("Pronto. Modelo padrão:", MODELO, "| documentos na base:", len(DOCS))

## Common retrieval post-processing techniques · PDF p. 350

#### 🧪 Exercício: Reordenar com MMR (diversidade)

Implemente *Maximal Marginal Relevance*: escolha k = 3 documentos equilibrando relevância para a consulta e diferença em relação aos já escolhidos (λ = 0,5). Use vetores TF-IDF para rodar sem o modelo.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
vec = TfidfVectorizer().fit([d["texto"] for d in DOCS])
D = vec.transform([d["texto"] for d in DOCS]).toarray(); q = vec.transform(["multa atraso livro"]).toarray()[0]
def cos(a, b): return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9))
def mmr(q, D, k=3, lam=0.5):
    escolhidos, resto = [], list(range(len(D)))
    while len(escolhidos) < k:
        melhor = max(resto, key=lambda i: lam * cos(q, D[i]) - (1 - lam) * max([cos(D[i], D[j]) for j in escolhidos], default=0))
        escolhidos.append(melhor); resto.remove(melhor)
    return escolhidos
print("só relevância:", [DOCS[i]["id"] for i in np.argsort([-cos(q, d) for d in D])[:3]])
print("MMR:", [DOCS[i]["id"] for i in mmr(q, D)])

## Difference between LangChain and LlamaIndex recency weighting · PDF p. 374

#### 🧪 Exercício: Peso por recência

Os documentos d04 e d05 falam de multa, mas d05 é a versão antiga. Combine similaridade (TF-IDF) com um peso de recência `0,99 ** dias_desde_atualizacao/30` e mostre que d04 passa à frente.

In [ ]:
from datetime import date
from sklearn.feature_extraction.text import TfidfVectorizer
vec = TfidfVectorizer().fit([d["texto"] for d in DOCS])
q = vec.transform(["multa por atraso"]).toarray()[0]
hoje = date(2025, 6, 1)
for d in DOCS:
    if d["id"] in ("d04", "d05"):
        v = vec.transform([d["texto"]]).toarray()[0]; sim = float(v @ q / (np.linalg.norm(v) * np.linalg.norm(q)))
        dias = (hoje - date.fromisoformat(d["data"])).days
        print(d["id"], round(sim, 3), round(sim * 0.99 ** (dias / 30), 3))

## Compression · PDF p. 375

#### 🧪 Exercício: Comprimir o contexto

Peça ao modelo para extrair, de cada um dos 3 documentos recuperados, **só as frases** relevantes para a pergunta "quantos livros posso pegar e por quanto tempo?". Compare os tokens antes e depois.

In [ ]:
q = "quantos livros posso pegar e por quanto tempo?"
recuperados = [DOCS[2], DOCS[3], DOCS[6]]
comprimido = [perguntar(f"Copie apenas as frases que ajudam a responder '{q}'. Se nenhuma ajudar, responda NADA.\n\n{d['texto']}") for d in recuperados]
antes = sum(contar_tokens(d["texto"]) for d in recuperados); depois = sum(contar_tokens(c) for c in comprimido if c.strip() != "NADA")
print(comprimido); print(antes, "->", depois, "tokens")

## Correction · PDF p. 385

#### 🧪 Exercício: Avaliar a recuperação antes de responder

Implemente um passo de correção (ideia do CRAG): para cada documento recuperado, peça ao modelo "relevante" ou "irrelevante"; se nenhum for relevante, responda "não encontrei na base" em vez de inventar.

In [ ]:
def corrigir(q, docs):
    bons = [d for d in docs if "irrelevante" not in perguntar(f"O texto ajuda a responder '{q}'? Responda relevante ou irrelevante.\n\n{d['texto']}", temperature=0).lower()]
    return bons or None
for q in ["qual o preço da impressão colorida?", "a biblioteca vende café?"]:
    bons = corrigir(q, DOCS[9:12])
    print(q, "->", [d["titulo"] for d in bons] if bons else "não encontrei na base")